# 01 - Ví dụ từng bước: Wisconsin Diagnostic Breast Cancer

Notebook minh họa tải/đọc dữ liệu, tách train/test có stratify, huấn luyện baseline và quy tắc hard voting. Nhãn dương tính `M` (malignant) được mã hóa là 1. Đây chỉ là thực nghiệm học thuật, không dùng để chẩn đoán.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocess import download_dataset, load_dataset, prepare_dataset

data_path = PROJECT_ROOT / "data" / "raw" / "wdbc.csv"
if not data_path.exists():
    download_dataset(data_path)
raw = load_dataset(data_path)
X, y, sample_ids = prepare_dataset(raw)
print(f"Mẫu: {len(X)}, đặc trưng: {X.shape[1]}")
print("Số lượng nhãn (0=benign, 1=malignant):")
display(y.value_counts().sort_index())

Mẫu: 569, đặc trưng: 30
Số lượng nhãn (0=benign, 1=malignant):


Diagnosis
0    357
1    212
Name: count, dtype: int64

## 1. Chia dữ liệu và huấn luyện baseline

Stratify giữ gần như nguyên tỷ lệ hai nhãn trong train và test. Imputation và scaling nằm bên trong pipeline nên chỉ học từ train.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from src.model import build_models

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
models = build_models(random_state=42)
baseline = models["Decision Tree"]
baseline.fit(X_train, y_train)
prediction = baseline.predict(X_test)
positive_probability = baseline.predict_proba(X_test)[:, 1]
print(f"Accuracy: {accuracy_score(y_test, prediction):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, positive_probability):.3f}")
print(classification_report(y_test, prediction, target_names=["Benign", "Malignant"], zero_division=0))

Accuracy: 0.904
ROC-AUC: 0.834
              precision    recall  f1-score   support

      Benign       0.90      0.96      0.93        72
   Malignant       0.92      0.81      0.86        42

    accuracy                           0.90       114
   macro avg       0.91      0.88      0.89       114
weighted avg       0.90      0.90      0.90       114



## 2. Hard voting tự cài đặt

Với mỗi mẫu, đếm số phiếu dự đoán của từng lớp và chọn lớp nhiều phiếu nhất. Nếu hòa, hàm chọn nhãn nhỏ hơn; với quy ước này là benign (0).

In [3]:
from src.model import majority_vote

toy_predictions = [
    [0, 1, 1],  # 2 phiếu malignant -> 1
    [0, 0, 1],  # 2 phiếu benign -> 0
    [0, 1, 0],  # 2 phiếu benign -> 0
]
majority_vote(toy_predictions)

array([1, 0, 0])

Có thể chạy toàn bộ so sánh baseline/ensemble, xuất bảng metric, ROC, confusion matrix và feature importance bằng `python main.py` hoặc notebook `02_application_experiments.ipynb`.